# AI Weather Forecast for KP Region
**Karakoram / Khyber Pakhtunkhwa (31-37N, 69-75E)**

This notebook uses NVIDIA Earth2Studio to run AI weather models on a GPU.
It generates forecast NetCDF files that you can download and load into the Streamlit dashboard.

**Requirements:** GPU runtime (Runtime > Change runtime type > T4 GPU)

In [ ]:
# Cell 1: Install Earth2Studio with model support
!pip install -q "earth2studio[data]" torch
# For FourCastNet model:
!pip install -q "earth2studio[fcn]"
# Verify GPU is available
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 2: Configuration
import numpy as np
from datetime import datetime, timedelta

# ===== CONFIGURE THESE =====
INIT_TIME = "2026-03-01T00:00:00"  # Forecast start date
N_STEPS = 40                        # 40 steps x 6hrs = 10 days
N_ENSEMBLE = 20                     # Number of ensemble members

# KP / Karakoram Region
KP_LAT_MIN, KP_LAT_MAX = 31.0, 37.0
KP_LON_MIN, KP_LON_MAX = 69.0, 75.0

# Variables to forecast
VARIABLES = ["t2m", "tp", "u10m", "v10m", "r2m", "msl"]
# ===========================

print(f"Forecast init: {INIT_TIME}")
print(f"Steps: {N_STEPS} ({N_STEPS * 6} hours / {N_STEPS * 6 / 24:.1f} days)")
print(f"Ensemble members: {N_ENSEMBLE}")
print(f"Region: {KP_LAT_MIN}-{KP_LAT_MAX}N, {KP_LON_MIN}-{KP_LON_MAX}E")

In [ ]:
# Cell 3: Load AI Weather Model
from earth2studio.models.px import FCN

print("Downloading FourCastNet model (first time only)...")
package = FCN.load_default_package()
model = FCN.load_model(package)
print(f"Model loaded: {model.__class__.__name__}")
print(f"Time step: 6 hours")

In [ ]:
# Cell 4: Set up data source and perturbation method
from earth2studio.data import GFS
from earth2studio.perturbation import SphericalGaussian
from earth2studio.io import ZarrBackend
import os

# GFS provides initial conditions (analysis fields)
data_source = GFS()

# Perturbation for ensemble spread
perturbation = SphericalGaussian(noise_amplitude=0.15)

# Output storage
os.makedirs("outputs", exist_ok=True)
io_backend = ZarrBackend(
    file_name="outputs/kp_ai_forecast.zarr",
    backend_kwargs={"overwrite": True}
)

print("Data source: GFS (NOAA)")
print("Perturbation: Spherical Gaussian (amplitude=0.15)")
print("Output: outputs/kp_ai_forecast.zarr")

In [ ]:
# Cell 5: Run Ensemble Inference
from earth2studio.run import ensemble as run_ensemble
import time

print(f"Starting {N_ENSEMBLE}-member ensemble forecast...")
print(f"This will take a few minutes on a T4 GPU.")

start = time.time()

io_result = run_ensemble(
    time=[INIT_TIME],
    nsteps=N_STEPS,
    nensemble=N_ENSEMBLE,
    model=model,
    data=data_source,
    io=io_backend,
    perturbation=perturbation,
    batch_size=2,  # Process 2 members at a time (fits on T4)
    output_coords={"variable": np.array(VARIABLES)}
)

elapsed = time.time() - start
print(f"Inference complete in {elapsed:.0f} seconds ({elapsed/60:.1f} minutes)")

In [ ]:
# Cell 6: Subset to KP region and save as NetCDF
import xarray as xr

print("Loading Zarr output...")
ds = xr.open_zarr("outputs/kp_ai_forecast.zarr")
print(f"Full dataset: {ds}")

# Subset to KP region
ds_kp = ds.sel(
    lat=slice(KP_LAT_MAX, KP_LAT_MIN),
    lon=slice(KP_LON_MIN, KP_LON_MAX)
)

# Save as NetCDF for download
output_path = "outputs/kp_weather_forecast.nc"
ds_kp.to_netcdf(output_path)
file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"Saved: {output_path} ({file_size:.1f} MB)")
print(f"KP region dataset:\n{ds_kp}")

In [ ]:
# Cell 7: Quick Visualization (Sanity Check)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Pick a mid-range lead time (day 5 = step 20)
step = min(20, N_STEPS - 1)

variables_to_plot = {
    "t2m": ("Temperature (K)", "RdBu_r"),
    "tp": ("Precipitation (m)", "Blues"),
    "u10m": ("U-Wind (m/s)", "coolwarm"),
    "v10m": ("V-Wind (m/s)", "coolwarm"),
    "r2m": ("Humidity (%)", "YlGn"),
    "msl": ("Pressure (Pa)", "coolwarm"),
}

for ax, (var, (label, cmap)) in zip(axes.flat, variables_to_plot.items()):
    if var in ds_kp:
        data = ds_kp[var].mean(dim="ensemble").isel(time=0)
        if "lead_time" in data.dims:
            data = data.isel(lead_time=step)
        data.squeeze().plot(ax=ax, cmap=cmap, add_colorbar=True)
        ax.set_title(f"Ensemble Mean: {label}")

plt.suptitle(f"AI Forecast: KP Region, Lead={step*6}hrs", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Download the forecast file
from google.colab import files

print("Downloading forecast NetCDF...")
print("Upload this file to the Streamlit dashboard using the sidebar uploader.")
files.download("outputs/kp_weather_forecast.nc")